# Fake News Detector - Corretivo

## 1. Instalação das Dependências

```bash
!pip install --upgrade --force-reinstall datasets transformers scikit-learn
```

In [1]:
!pip install --upgrade --force-reinstall datasets transformers scikit-learn

In [2]:
# 2. Verificar possíveis conflitos de módulo `datasets`
import os
conflict = os.path.exists('datasets.py') or os.path.isdir('datasets')
print(f"Conflito de nome detectado? {conflict}")

Conflito de nome detectado? False


In [3]:
# 3. Teste de importação
try:
    import datasets
    from datasets import load_dataset
    print('Importação bem-sucedida do datasets da Hugging Face')
except Exception as e:
    print('Erro na importação do datasets:', e)

Importação bem-sucedida do datasets da Hugging Face


In [ ]:
# 4. Carregar dataset Liar
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

dataset = load_dataset('liar')
print('Split de treino:', dataset['train'].num_rows)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

liar.py:   0%|          | 0.00/6.41k [00:00<?, ?B/s]

## 5. Pré-processamento e Tokenização

In [ ]:
label_mapping = {
    'pants-fire': 0,
    'false': 1,
    'barely-true': 2,
    'half-true': 3,
    'mostly-true': 4,
    'true': 5
}
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    examples['label'] = [label_mapping[label] for label in examples['label']]
    return tokenizer(examples['statement'], truncation=True, padding='max_length')

tokenized = dataset.map(preprocess_function, batched=True)

## 6. Configuração do Modelo e Treinamento

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=6)
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir='./logs',
)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

## 7. Avaliação e Inferência

In [ ]:
results = trainer.evaluate()
print('Resultados de avaliação:', results)

In [ ]:
def predict(texts):
    inputs = tokenizer(texts, return_tensors='pt', truncation=True, padding='max_length')
    outputs = model(**inputs)
    preds = np.argmax(outputs.logits.detach().numpy(), axis=1)
    inv_map = {v: k for k, v in label_mapping.items()}
    return [inv_map[p] for p in preds]

exs = [
    "The senator secretly met with aliens last night.",
    "New study shows coffee reduces risk of type 2 diabetes."
]
print('Predições de exemplo:', predict(exs))